# Data leakage

Flujo con y sin data leakage para el parametro AUC

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC
from sklearn.preprocessing import MinMaxScaler, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score

# Cargar datos
df = pd.read_csv("../data/heart.csv")

# Encodear categóricas
categoricas = df.select_dtypes(include='object').columns.tolist()
encoder = OrdinalEncoder()
df[categoricas] = encoder.fit_transform(df[categoricas])

X = df.drop("HeartDisease", axis=1)
y = df["HeartDisease"]

# Variable artificial que introduce fuga
np.random.seed(0)
X_leaky = X.copy()
X_leaky["leaky_feature"] = y + np.random.normal(0, 0.01, size=len(y))

# Flujo CON fuga (usa X_leaky con la variable falsa)
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_leaky)
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

grid_l = GridSearchCV(SVC(probability=True), param_grid={"C": [0.1, 1, 10], "gamma": [0.01, 0.1]}, cv=5, scoring="roc_auc")
grid_l.fit(X_train_l, y_train_l)
auc_l = roc_auc_score(y_test_l, grid_l.predict_proba(X_test_l)[:, 1])

# Flujo SIN fuga (usa X original sin leaky_feature)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

pipe = Pipeline([("scaler", MinMaxScaler()), ("svc", SVC(probability=True))])
param_grid = {"svc__C": [0.1, 1, 10], "svc__gamma": [0.01, 0.1]}
grid = GridSearchCV(pipe, param_grid, cv=5, scoring="roc_auc")
grid.fit(X_train, y_train)
auc = roc_auc_score(y_test, grid.predict_proba(X_test)[:, 1])

print(f"AUC con fuga: {auc_l:.3f}")
print(f"AUC sin fuga: {auc:.3f}")

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_27552\2703947174.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categoricas = df.select_dtypes(include='object').columns.tolist()


AUC con fuga: 1.000
AUC sin fuga: 0.903


Se puede ver como el AUC con fuga puede ser engañoso porque muestra un modelo perfecto mientras que el sin fuga muestra un AUC del 0.903 el cual es correcto